# SVM vs Random Forest - CICIDS2017 (CTGAN Balanced - GPU)
Benchmark on cleaned CICIDS2017 dataset with CTGAN data augmentation and 70-30 train/test split

**Strategy:**
- Majority classes (>10,286 samples): Undersample to 50,000
- Minority classes (≤10,286 samples): CTGAN augment to 20,000
- CTGAN runs on GPU for faster training

In [4]:
!pip install ctgan imbalanced-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.9 MB/s eta 0:00:0000:01


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! CTGAN will run on CPU (much slower).")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from ctgan import CTGAN
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

USE_GPU = torch.cuda.is_available()
print(f"Libraries imported successfully!")
print(f"CTGAN will use: {'GPU ✓' if USE_GPU else 'CPU ✗'}")

Libraries imported successfully!
CTGAN will use: GPU ✓


## Step 1: Mount Drive & Load Data

In [6]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/data_processed/cicids2017_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nOriginal Class Distribution:")
print(df['Label'].value_counts())

Mounted at /content/drive
Dataset shape: (2520798, 79)

Original Class Distribution:
Label
0     2095057
4      172846
2      128014
10      90694
3       10286
7        5931
6        5385
5        5228
11       3219
1        1948
12       1470
14        652
9          36
13         21
8          11
Name: count, dtype: int64


## Step 2: Train/Test Split (70-30)

In [7]:
X = df.drop('Label', axis=1)
y = df['Label']
feature_columns = X.columns.tolist()

print(f"Total features: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution (BEFORE balancing):")
print(y_train.value_counts())

Total features: 78
Total samples: 2520798

Train set: 1764558 samples
Test set: 756240 samples

Training set class distribution (BEFORE balancing):
Label
0     1466539
4      120992
2       89610
10      63486
3        7200
7        4152
6        3769
5        3660
11       2253
1        1364
12       1029
14        456
9          25
13         15
8           8
Name: count, dtype: int64


## Step 3: Undersample Majority Classes to 50,000

In [8]:
threshold = 10286
target_minority = 20000

under_strategy = {}
for label, count in Counter(y_train).items():
    if count > threshold:
        under_strategy[label] = 50000

print(f"Undersampling classes {list(under_strategy.keys())} -> 50,000 each")

under_sampler = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)

print(f"After undersampling: {len(y_train_under):,} samples")
print(pd.Series(y_train_under).value_counts())

Undersampling classes [4, 0, 2, 10] -> 50,000 each
After undersampling: 223,931 samples
Label
0     50000
2     50000
4     50000
10    50000
3      7200
7      4152
6      3769
5      3660
11     2253
1      1364
12     1029
14      456
9        25
13       15
8         8
Name: count, dtype: int64


## Step 4: CTGAN Augmentation on GPU (Minority Classes → 20,000)

In [9]:
class_counts = Counter(y_train_under)
minority_classes = {label: count for label, count in class_counts.items() if count <= threshold}

print("=" * 60)
print(f"CTGAN DATA AUGMENTATION ({'GPU' if USE_GPU else 'CPU'})")
print("=" * 60)
print(f"\nTarget for minority classes: {target_minority}")
print(f"\nClasses to augment:")
for label, count in sorted(minority_classes.items()):
    print(f"  Label {label}: {count:,} -> {target_minority:,} (+{target_minority - count:,} synthetic)")

CTGAN DATA AUGMENTATION (GPU)

Target for minority classes: 20000

Classes to augment:
  Label 1: 1,364 -> 20,000 (+18,636 synthetic)
  Label 3: 7,200 -> 20,000 (+12,800 synthetic)
  Label 5: 3,660 -> 20,000 (+16,340 synthetic)
  Label 6: 3,769 -> 20,000 (+16,231 synthetic)
  Label 7: 4,152 -> 20,000 (+15,848 synthetic)
  Label 8: 8 -> 20,000 (+19,992 synthetic)
  Label 9: 25 -> 20,000 (+19,975 synthetic)
  Label 11: 2,253 -> 20,000 (+17,747 synthetic)
  Label 12: 1,029 -> 20,000 (+18,971 synthetic)
  Label 13: 15 -> 20,000 (+19,985 synthetic)
  Label 14: 456 -> 20,000 (+19,544 synthetic)


In [11]:
train_df = pd.DataFrame(X_train_under, columns=feature_columns)
train_df['Label'] = y_train_under.values

synthetic_dfs = []

for label, current_count in sorted(minority_classes.items()):
    samples_needed = target_minority - current_count
    if samples_needed <= 0:
        print(f"Label {label}: Already has {current_count} samples, skipping.")
        continue

    print(f"\n{'='*50}")
    print(f"CTGAN for Label {label}: {current_count} -> {target_minority} (+{samples_needed} synthetic)")
    print(f"{'='*50}")

    class_data = train_df[train_df['Label'] == label].drop('Label', axis=1)

    # Train CTGAN with GPU enabled
    ctgan = CTGAN(
        epochs=300,
        batch_size=max(2, min(500, current_count) // 2 * 2),
        pac=1,
        cuda=USE_GPU,
        verbose=True
    )
    ctgan.fit(class_data)

    synthetic = ctgan.sample(samples_needed)
    synthetic['Label'] = label
    synthetic_dfs.append(synthetic)

    print(f"Label {label}: Generated {len(synthetic)} synthetic samples ✓")

    # Free GPU memory after each class
    if USE_GPU:
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All CTGAN augmentation complete!")
print(f"{'='*60}")


CTGAN for Label 1: 1364 -> 20000 (+18636 synthetic)


Gen. (-00.18) | Discrim. (-00.26): 100%|██████████| 300/300 [00:50<00:00,  5.99it/s]


Label 1: Generated 18636 synthetic samples ✓

CTGAN for Label 3: 7200 -> 20000 (+12800 synthetic)


Gen. (+00.01) | Discrim. (-00.38): 100%|██████████| 300/300 [05:47<00:00,  1.16s/it]


Label 3: Generated 12800 synthetic samples ✓

CTGAN for Label 5: 3660 -> 20000 (+16340 synthetic)


...

Gen. (-00.09) | Discrim. (-00.08): 100%|██████████| 300/300 [03:17<00:00,  1.52it/s]


Label 7: Generated 15848 synthetic samples ✓

CTGAN for Label 8: 8 -> 20000 (+19992 synthetic)


Gen. (-01.42) | Discrim. (+00.02): 100%|██████████| 300/300 [00:26<00:00, 11.48it/s]


Label 8: Generated 19992 synthetic samples ✓

CTGAN for Label 9: 25 -> 20000 (+19975 synthetic)


AssertionError: 

In [ ]:
if synthetic_dfs:
    all_synthetic = pd.concat(synthetic_dfs, ignore_index=True)
    balanced_df = pd.concat([train_df, all_synthetic], ignore_index=True)
else:
    balanced_df = train_df.copy()

X_train_balanced = balanced_df.drop('Label', axis=1).values
y_train_balanced = balanced_df['Label'].values

print("CTGAN BALANCED TRAINING SET:")
print(pd.Series(y_train_balanced).value_counts())
print(f"\nTotal training samples: {len(y_train_balanced):,}")

## Step 5: Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled successfully!")
print(f"Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

## Step 6: Train SVM (SGDClassifier - Hinge Loss)

In [ ]:
print("Training SVM (SGDClassifier with Hinge Loss)...")
svm_model = SGDClassifier(
    loss='hinge',
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    tol=1e-3,
    n_jobs=-1
)
svm_model.fit(X_train_scaled, y_train_balanced)

y_train_pred_svm = svm_model.predict(X_train_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

svm_metrics = {
    'Algorithm': 'SVM (SGDClassifier - Hinge Loss)',
    'Train Accuracy': accuracy_score(y_train_balanced, y_train_pred_svm),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_svm),
    'Precision': precision_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_svm, average='weighted', zero_division=0)
}

print(f"SVM Training Complete!")
print(f"  Train Accuracy: {svm_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {svm_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {svm_metrics['Precision']:.4f}")
print(f"  Recall: {svm_metrics['Recall']:.4f}")
print(f"  F1-Score: {svm_metrics['F1-Score']:.4f}")

## Step 7: Train Random Forest

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample'
)
rf_model.fit(X_train_balanced, y_train_balanced)

y_train_pred_rf = rf_model.predict(X_train_balanced)
y_test_pred_rf = rf_model.predict(X_test)

rf_metrics = {
    'Algorithm': 'Random Forest',
    'Train Accuracy': accuracy_score(y_train_balanced, y_train_pred_rf),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_rf),
    'Precision': precision_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_rf, average='weighted', zero_division=0)
}

print(f"Random Forest Training Complete!")
print(f"  Train Accuracy: {rf_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {rf_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {rf_metrics['Precision']:.4f}")
print(f"  Recall: {rf_metrics['Recall']:.4f}")
print(f"  F1-Score: {rf_metrics['F1-Score']:.4f}")

## Step 8: Compare Results

In [ ]:
comparison_df = pd.DataFrame([svm_metrics, rf_metrics])
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON (CTGAN Balanced Dataset)")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

## Step 9: Visualize Metrics (Line Plot)

In [ ]:
metrics_data = comparison_df.set_index('Algorithm').T

fig, ax = plt.subplots(figsize=(12, 6))
metrics_data.plot(ax=ax, marker='o', linewidth=2, markersize=8)
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Metrics', fontsize=12)
ax.set_title('SVM vs Random Forest - Performance Metrics (CTGAN Balanced)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

for col in metrics_data.columns:
    for idx, val in enumerate(metrics_data[col]):
        ax.text(idx, val + 0.02, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 10: Visualize Metrics (Bar Plot)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = ['Train Accuracy', 'Test Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#1f77b4', '#ff7f0e']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = [svm_metrics[metric], rf_metrics[metric]]
    bars = ax.bar(['SVM', 'Random Forest'], values, color=colors, alpha=0.8, edgecolor='black')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_ylim([0, 1.05])
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.delaxes(axes[5])
fig.suptitle('SVM vs Random Forest - Detailed Metrics (CTGAN Balanced)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Step 11: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_svm = confusion_matrix(y_test, y_test_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('SVM Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_rf = confusion_matrix(y_test, y_test_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=True)
axes[1].set_title('Random Forest Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 12: Classification Reports

In [ ]:
print("\n" + "="*80)
print("SVM - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_svm, zero_division=0))

print("\n" + "="*80)
print("RANDOM FOREST - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_rf, zero_division=0))

## Step 13: Summary

In [ ]:
print("\n" + "#"*80)
print("# SUMMARY")
print("#"*80)

best_f1 = 'SVM' if svm_metrics['F1-Score'] > rf_metrics['F1-Score'] else 'Random Forest'
best_acc = 'SVM' if svm_metrics['Test Accuracy'] > rf_metrics['Test Accuracy'] else 'Random Forest'

print(f"\nDataset: CICIDS2017 (Cleaned + CTGAN Balanced)")
print(f"Total Samples (Original): {len(df):,}")
print(f"Total Training Samples (After CTGAN): {len(y_train_balanced):,}")
print(f"Total Features: {X.shape[1]}")
print(f"Train/Test Split: 70/30")
print(f"Test Samples: {len(X_test):,}")
print(f"\nBalancing Strategy:")
print(f"  - Classes > {threshold}: Undersampled to 50,000")
print(f"  - Classes <= {threshold}: CTGAN augmented to {target_minority}")

print(f"\n" + "-"*80)
print(f"BEST MODEL (by Test Accuracy): {best_acc}")
print(f"BEST MODEL (by F1-Score): {best_f1}")
print(f"-"*80)

print(f"\nSVM Performance:")
for key, value in svm_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\nRandom Forest Performance:")
for key, value in rf_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\n" + "#"*80)

## Step 14: Save Models to Drive

In [ ]:
import joblib
import os

save_path = '/content/drive/MyDrive/IDS_Project/saved_models_ctgan'
os.makedirs(save_path, exist_ok=True)

joblib.dump(svm_model, f'{save_path}/svm_model_ctgan.pkl')
joblib.dump(rf_model, f'{save_path}/rf_model_ctgan.pkl')
joblib.dump(scaler, f'{save_path}/scaler_ctgan.pkl')

print("All models saved to Google Drive!")
print(f"Location: {save_path}")